# 06 - Explicabilidad (XAI)

Fase 3 del TFM: generar mapas Grad-CAM para los mejores modelos de clasificacion y cuantificar si la saliencia cae dentro de las mascaras disponibles.

- CXR: la mascara es pulmonar, por tanto mide plausibilidad anatomica, no lesion COVID.
- CT: la mascara es de infeccion/lesion, disponible solo en los estudios anotados de MosMedData.

Este notebook usa el script reproducible `scripts/generate_xai_explanations.py`.

In [ ]:

from pathlib import Path
import json
import os
import subprocess
import sys

import pandas as pd
from IPython.display import Image, display

KNOWN_PROJECT_ROOT = Path('/Users/yuncaichen/Master/TFM:FINAL/TFM')

def find_project_root(start: Path) -> Path:
    candidates = [start, *start.parents, KNOWN_PROJECT_ROOT]
    for candidate in candidates:
        if (candidate / 'src' / 'config.py').exists():
            return candidate
    raise FileNotFoundError('No se encontro la raiz del proyecto con src/config.py')

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
os.environ.setdefault('MPLCONFIGDIR', str(PROJECT_ROOT / 'results' / '.matplotlib'))

print(f'Proyecto cargado desde: {PROJECT_ROOT}')



## Configuracion

Por defecto se explican los modelos consolidados:

- CXR: `cxr_densenet121_weighted_ce_full`, el mejor clasificador CXR por rendimiento global.
- CT: `ct_densenet121_baseline_full`, el mejor clasificador CT por F1-macro a nivel de slice dentro de la matriz original.

En CT hay dos lecturas posibles:

- `CT_MASK_SPLIT = 'test'`: mas conservador, porque usa solo mascaras de estudios que caen en el test split de clasificacion. En tu caso genera pocos ejemplos.
- `CT_MASK_SPLIT = 'all'`: mas util para inspeccion cualitativa, porque usa todas las mascaras CT disponibles, pero no debe presentarse como evaluacion final independiente.

Grad-CAM se calcula sobre la **clase predicha por el modelo**, porque aqui queremos explicar la decision tomada por el clasificador. Si el modelo se equivoca, el mapa explica el error, no la clase correcta.


In [ ]:

RUN_XAI = True
DATASET = 'both'  # 'cxr', 'ct' o 'both'
MAX_PER_CLASS = '2'
MAX_INCORRECT_PER_CLASS = '1'
CT_MASK_SPLIT = 'all'  # 'test' para evaluacion conservadora; 'all' para inspeccion cualitativa amplia

if RUN_XAI:
    subprocess.run(
        [
            str(PROJECT_ROOT / 'scripts' / 'run_xai_explainability.sh'),
            DATASET,
            MAX_PER_CLASS,
            MAX_INCORRECT_PER_CLASS,
            CT_MASK_SPLIT,
        ],
        cwd=PROJECT_ROOT,
        check=True,
    )



## Lectura de metricas XAI

Esta parte resume los CSV generados por Grad-CAM. Las metricas no se deben leer como rendimiento diagnostico, sino como alineamiento espacial entre saliencia y mascara disponible.

- `saliency_mask_iou`: solapamiento entre la region saliente binarizada y la mascara.
- `saliency_inside_mask_ratio`: proporcion de intensidad Grad-CAM que cae dentro de la mascara.
- `saliency_peak_inside_mask`: indica si el punto de maxima saliencia cae dentro de la mascara.

En CXR la mascara es pulmonar, por tanto mide plausibilidad anatomica. En CT la mascara representa afectacion/infeccion, pero solo en el subconjunto anotado.


In [ ]:

xai_root = PROJECT_ROOT / 'results' / 'explainability'
all_metric_paths = sorted(xai_root.glob('*/*/*_xai_metrics.csv'))
all_summary_paths = sorted(xai_root.glob('*/*/*_xai_summary.json'))

def infer_mask_split(path: Path) -> str:
    folder = path.parent.name
    if folder.endswith('_all'):
        return 'all'
    if folder.endswith('_test'):
        return 'test'
    return 'test'

def keep_xai_path(path: Path) -> bool:
    dataset = path.relative_to(xai_root).parts[0]
    folder = path.parent.name
    if dataset == 'ct' and not folder.endswith(('_all', '_test')):
        return False
    return True

metric_paths = [path for path in all_metric_paths if keep_xai_path(path)]
summary_paths = [path for path in all_summary_paths if keep_xai_path(path)]

metric_frames = []
for path in metric_paths:
    frame = pd.read_csv(path)
    frame['mask_split'] = infer_mask_split(path)
    frame['metrics_path'] = str(path)
    metric_frames.append(frame)
metrics_df = pd.concat(metric_frames, ignore_index=True) if metric_frames else pd.DataFrame()

summary_rows = []
for path in summary_paths:
    row = json.loads(path.read_text())
    row['mask_split'] = infer_mask_split(path)
    row['summary_path'] = str(path)
    summary_rows.append(row)
summary_df = pd.DataFrame(summary_rows)

summary_cols = [
    'dataset', 'experiment', 'architecture', 'mask_split', 'mask_reference',
    'num_examples', 'saliency_quantile', 'mean_saliency_mask_iou',
    'mean_saliency_inside_mask_ratio', 'peak_inside_mask_rate', 'note'
]
if not summary_df.empty:
    display(summary_df[summary_cols].sort_values(['dataset', 'experiment', 'mask_split']).round(4))
else:
    print('No hay summaries XAI generados todavia.')

if not metrics_df.empty:
    per_example_cols = [
        'dataset', 'mask_split', 'sample_id', 'true_label', 'pred_label', 'is_correct',
        'confidence', 'saliency_mask_iou', 'saliency_inside_mask_ratio',
        'saliency_peak_inside_mask'
    ]
    display(metrics_df[per_example_cols].sort_values(['dataset', 'mask_split', 'sample_id']).round(4))
else:
    print('No hay metricas XAI por ejemplo generadas todavia.')



## Visualizacion de ejemplos generados

Para evitar confusiones, se muestran ejemplos separados por modalidad. En cada figura:

1. `Imagen`: entrada original.
2. `Mascara`: pulmon en CXR o afectacion/infeccion en CT.
3. `Grad-CAM`: mapa continuo de saliencia.
4. `Saliencia binaria`: region Grad-CAM tras umbralizar el mapa.
5. `Mascara vs saliencia`: comparacion visual entre referencia y region saliente.


In [ ]:

for dataset_name in ['cxr', 'ct']:
    figure_paths = sorted((xai_root / dataset_name).glob('*/figures/*_gradcam.png'))
    print(f'\n=== {dataset_name.upper()} | {len(figure_paths)} figuras disponibles ===')
    for path in figure_paths[:6]:
        print(path.relative_to(PROJECT_ROOT))
        display(Image(filename=str(path)))



## Interpretacion para el informe

En esta fase no buscamos demostrar que Grad-CAM sea una segmentacion. Grad-CAM solo aproxima que zonas influyen en la decision del clasificador.

La interpretacion actual de tus resultados es:

- En CXR, la saliencia se alinea parcialmente con los pulmones. Esto aporta plausibilidad anatomica, pero no demuestra localizacion de lesion COVID porque la mascara disponible es pulmonar.
- En CT, el alineamiento con las mascaras de afectacion es muy bajo. Esto sugiere que el clasificador de severidad puede usar informacion global o contextual del slice, y no necesariamente la region patologica anotada.
- El numero de ejemplos CT en `test` es pequeno porque las mascaras CT disponibles son limitadas y se concentran en estudios CT-1. Para figuras cualitativas puede usarse `CT_MASK_SPLIT = 'all'`, dejando claro que no es una evaluacion final independiente.

Frase defendible: un buen resultado de clasificacion no implica automaticamente que el modelo base su decision en la region clinicamente esperada; por eso Grad-CAM debe interpretarse como auditoria visual y no como prueba causal.
